BAUDE-131


In [0]:
%sql
-- 1. νέα int στήλη
ALTER TABLE `p-cdh`.primaryactive.exposure
  ADD COLUMN DaysPastDueWithMateriality_int int AFTER MaterialityStartDate;

-- 2. backfill: qca/arotron -> από DaysPastDue, αλλιώς κράτα την υπάρχουσα τιμή
UPDATE `p-cdh`.primaryactive.exposure
  SET DaysPastDueWithMateriality_int =
        CASE WHEN SourceSystem IN ('qca','arotron')
             THEN CAST(DaysPastDue AS int)
             ELSE CAST(DaysPastDueWithMateriality AS int)
        END;

-- 3. μηδενισμός του DaysPastDue για qca/arotron
UPDATE `p-cdh`.primaryactive.exposure
  SET DaysPastDue = NULL
  WHERE SourceSystem IN ('qca','arotron');

-- 4. drop παλιάς decimal + rename
ALTER TABLE `p-cdh`.primaryactive.exposure
  DROP COLUMN DaysPastDueWithMateriality;

ALTER TABLE `p-cdh`.primaryactive.exposure
  RENAME COLUMN DaysPastDueWithMateriality_int TO DaysPastDueWithMateriality;

In [0]:
%sql
-- 1. νέα int στήλη
ALTER TABLE `p-cdh`.primaryarchive.exposure
  ADD COLUMN DaysPastDueWithMateriality_int int AFTER MaterialityStartDate;

-- 2. backfill: qca/arotron -> από DaysPastDue, αλλιώς κράτα την υπάρχουσα τιμή
UPDATE `p-cdh`.primaryarchive.exposure
  SET DaysPastDueWithMateriality_int =
        CASE WHEN SourceSystem IN ('qca','arotron')
             THEN CAST(DaysPastDue AS int)
             ELSE CAST(DaysPastDueWithMateriality AS int)
        END;

-- 3. μηδενισμός του DaysPastDue για qca/arotron
UPDATE `p-cdh`.primaryarchive.exposure
  SET DaysPastDue = NULL
  WHERE SourceSystem IN ('qca','arotron');

-- 4. drop παλιάς decimal + rename
ALTER TABLE `p-cdh`.primaryarchive.exposure
  DROP COLUMN DaysPastDueWithMateriality;

ALTER TABLE `p-cdh`.primaryarchive.exposure
  RENAME COLUMN DaysPastDueWithMateriality_int TO DaysPastDueWithMateriality;

BAUDE-131 CUD UPDATES

In [0]:
%sql
-- 1. νέα int στήλη
ALTER TABLE `p-cdh`.derivativeactive.dexposure
  ADD COLUMN DaysPastDueWithMateriality_int int AFTER MaterialityStartDate;

-- 2. backfill: qca/arotron -> από DaysPastDue, αλλιώς κράτα την υπάρχουσα τιμή
UPDATE `p-cdh`.derivativeactive.dexposure
  SET DaysPastDueWithMateriality_int =
        CASE WHEN SourceSystem IN ('qca','arotron')
             THEN CAST(DaysPastDue AS int)
             ELSE CAST(DaysPastDueWithMateriality AS int)
        END;

-- 3. μηδενισμός του DaysPastDue για qca/arotron
UPDATE `p-cdh`.derivativeactive.dexposure
  SET DaysPastDue = NULL
  WHERE SourceSystem IN ('qca','arotron');

-- 4. drop παλιάς decimal + rename
ALTER TABLE `p-cdh`.derivativeactive.dexposure
  DROP COLUMN DaysPastDueWithMateriality;

ALTER TABLE `p-cdh`.derivativeactive.dexposure
  RENAME COLUMN DaysPastDueWithMateriality_int TO DaysPastDueWithMateriality;

In [0]:
%sql
-- 1. νέα int στήλη
ALTER TABLE `p-cdh`.derivativearchive.dexposure
  ADD COLUMN DaysPastDueWithMateriality_int int AFTER MaterialityStartDate;

-- 2. backfill: qca/arotron -> από DaysPastDue, αλλιώς κράτα την υπάρχουσα τιμή
UPDATE `p-cdh`.derivativearchive.dexposure
  SET DaysPastDueWithMateriality_int =
        CASE WHEN SourceSystem IN ('qca','arotron')
             THEN CAST(DaysPastDue AS int)
             ELSE CAST(DaysPastDueWithMateriality AS int)
        END;

-- 3. μηδενισμός του DaysPastDue για qca/arotron
UPDATE `p-cdh`.derivativearchive.dexposure
  SET DaysPastDue = NULL
  WHERE SourceSystem IN ('qca','arotron');

-- 4. drop παλιάς decimal + rename
ALTER TABLE `p-cdh`.derivativearchive.dexposure
  DROP COLUMN DaysPastDueWithMateriality;

ALTER TABLE `p-cdh`.derivativearchive.dexposure
  RENAME COLUMN DaysPastDueWithMateriality_int TO DaysPastDueWithMateriality;

correction_eom_orchestrator


In [0]:
%sql
UPDATE `p-cdh-delivery`.report_catalog.report_registry SET depends_on_workflow = 'Derivative Unity Run Targeted' WHERE report_name = 'AlphaBankGeneralli';

AML REPORT

In [0]:
%sql
MERGE INTO `p-cdh-delivery`.aml.aml_report AS target
USING (
    SELECT s.id AS sep18_id, r.id AS rerun_id
    FROM `d-cdh`.reference.`2026_09_18_run` s
    JOIN `p-cdh-delivery`.aml.aml_report r
      ON  s.Tax_Number  = r.Tax_Number
      AND s.rule_name   = r.rule_name
      AND s.date_to     = r.date_to
      AND s.date_from   = r.date_from
    WHERE r.report_date = (SELECT MAX(report_date) FROM `p-cdh-delivery`.aml.aml_report)
      ) AS mapping
ON target.id = mapping.rerun_id
WHEN MATCHED THEN UPDATE SET target.id = mapping.sep18_id

DNA26-1426


In [0]:
%sql

insert into `p-cdh-delivery`.report_catalog.report_registry(report_name,job_name,frequency,run_days,is_active,execution_order,depends_on_workflow,skip_email,created_at)
values  ('portfolio_cdi_bal','portfolio_cdi_bal','daily','Tuesday,Wednesday,Thursday,Friday,Saturday',true,10,'Derivative Job Run - 3rdparties - Ptools',false,getdate())
, ('ESB Payment Plan Job','ESB Payment Plan Job','daily','Tuesday,Wednesday,Thursday,Friday,Saturday',true,10,'Derivative Job Run - 3rdparties - Ptools',false,getdate())
, ('NPV Tool Job','NPV Tool Job','daily','Tuesday,Wednesday,Thursday,Friday,Saturday',true,10,'Derivative Job Run - 3rdparties - Ptools',false,getdate())
, ('pnrToolExport','pnrToolExport','daily','Tuesday,Wednesday,Thursday,Friday,Saturday',true,10,'Derivative Job Run - 3rdparties - Ptools',false,getdate())
, ('FLDT Application Job','FLDT Application Job','daily','Tuesday,Wednesday,Thursday,Friday,Saturday',true,10,'Derivative Job Run - 3rdparties - Ptools',false,getdate())
, ('RCG IT','RCG IT','daily','Tuesday,Wednesday,Thursday,Friday,Saturday',true,10,'Derivative Job Run - 3rdparties - Ptools',false,getdate())
, ('Platform IT','Platform IT','daily','Tuesday,Wednesday,Thursday,Friday,Saturday',true,10,'Derivative Job Run - 3rdparties - Ptools',false,getdate())